In [1]:
import pandas as pd
import numpy as np
import tarfile
import io
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

FILE_PATH_GZ = 'GCD_VMs.tar.gz'
TEST_SIZE_RATIO = 0.2
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
EPOCHS = 15
BATCH_SIZE = 256
LAG_STEPS = 20

all_data = []
with tarfile.open(FILE_PATH_GZ, 'r:gz') as tar:
    for member in tar.getmembers():
        if member.isfile():
            f = tar.extractfile(member)
            if f:
                df_vm = pd.read_csv(io.BytesIO(f.read()), sep=r"\s+", header=None,
                                   names=["CPU", "Memory"], engine="python")
                all_data.append(df_vm)

df_raw = pd.concat(all_data, ignore_index=True)

print("Dataset Info")
df_raw.info()

print("\nStatistical Summary")
print(df_raw.describe())

Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460800 entries, 0 to 460799
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   CPU     460800 non-null  float64
 1   Memory  460800 non-null  float64
dtypes: float64(2)
memory usage: 7.0 MB

Statistical Summary
                 CPU         Memory
count  460800.000000  460800.000000
mean       21.849631      19.558832
std        13.625046      16.665227
min         5.005300       5.043600
25%        12.170000       8.921900
50%        18.550000      12.434250
75%        27.041000      25.298212
max        89.367000     215.313651


In [2]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_raw[["CPU", "Memory"]])

In [3]:
def create_sequences(data, seq_len):
    x, y = [], []
    for i in range(len(data) - seq_len):
        x.append(data[i:i+seq_len])
        y.append(data[i+seq_len, 0])
    return np.array(x), np.array(y)

X, y = create_sequences(scaled_data, LAG_STEPS)

In [4]:
split_idx = int(len(X) * (1 - TEST_SIZE_RATIO))
x_train, x_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

In [5]:
class GCDDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_dataset = GCDDataset(x_train, y_train)
test_dataset = GCDDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [6]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_dim, hidden_dim // 2)
        self.out = nn.Linear(hidden_dim // 2, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        out, _ = self.lstm(x)

        x = out[:, -1, :]
        x = self.relu(self.fc(x))
        return self.out(x)

In [7]:
def train(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, device, scaler):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            all_preds.append(out.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    preds = np.concatenate(all_preds).flatten()
    targets = np.concatenate(all_targets).flatten()


    cpu_mean, cpu_std = scaler.mean_[0], scaler.scale_[0]
    preds = np.clip(preds * cpu_std + cpu_mean, 0, 100)
    targets = targets * cpu_std + cpu_mean

    return targets, preds

model = LSTMModel().to(DEVICE)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"{'Epoch':<10} | {'Train Loss':<12} | {'Val RMSE':<10} | {'Val MAE':<10}")
print("-" * 50)

for epoch in range(EPOCHS):
    train_loss = train(model, train_loader, optimizer, loss_fn, DEVICE)
    y_true, y_pred = evaluate(model, test_loader, DEVICE, scaler)

    epoch_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    epoch_mae = mean_absolute_error(y_true, y_pred)

    print(f"{epoch+1:<10} | {train_loss:<12.4f} | {epoch_rmse:<10.4f} | {epoch_mae:<10.4f}")

y_true, y_pred = evaluate(model, test_loader, DEVICE, scaler)

print("\n--- LSTM Results ---")
print(f"R²:   {r2_score(y_true, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.4f} % CPU")
print(f"MAE:  {mean_absolute_error(y_true, y_pred):.4f} % CPU")

Epoch      | Train Loss   | Val RMSE   | Val MAE   
--------------------------------------------------
1          | 0.0464       | 2.9421     | 1.6662    


KeyboardInterrupt: 